# Part 3 Transformer-Based Sentiment Analysis
Welcome to the state-of-the-art! In this notebook, we abandon traditional counting/statistics and sequence-level RNNs, and step into the world of **Transformers** using Hugging Face's `transformers` library.

We will:
1. Load a pretrained transformer (**BERT**).
2. Tokenize our text specifically for BERT.
3. Fine-tune BERT on our movie review dataset using the `Trainer` API.
4. Evaluate its performance and context understanding.

In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Ensure GPU is used if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# We take a SMALL SUBSET of the data for this notebook. 
# Fine-tuning BERT on all 25,000 reviews takes several hours on a standard machine.
print("Loading a subset of the dataset for speed...")
train_df = pd.read_csv('../data/imdb_train.csv').sample(5000, random_state=42)
test_df = pd.read_csv('../data/imdb_test.csv').sample(1500, random_state=42)

train_df['label'] = train_df['sentiment'].map({"Negative": 0, "Positive": 1})
test_df['label'] = test_df['sentiment'].map({"Negative": 0, "Positive": 1})

# Convert Pandas DataFrames into Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df[['text', 'label']])
test_dataset = Dataset.from_pandas(test_df[['text', 'label']])

Using device: cuda
Loading a subset of the dataset for speed...


## Step 1 & 2 Load Pretrained Tokenizer & Model
BERT understands text completely differently than TF-IDF. It breaks words into "sub-words" (e.g., "playing" -> "play", "##ing") and maps them to highly contextual mathematical vectors.

We use `bert-base-uncased` which means all text is converted to lowercase.

In [2]:
model_name = "bert-base-uncased"

print(f"Loading tokenizer: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model architecture...")
# We specify num_labels=2 because we have Positive (1) and Negative (0)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device)

def tokenize_function(examples):
    # Padding and truncation ensure all inputs are the exact same length (512 tokens max for BERT)
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=256)

print("Tokenizing training data...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)

print("Tokenizing testing data...")
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Loading tokenizer: bert-base-uncased
Loading model architecture...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tokenizing training data...


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing testing data...


Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [3]:
model2_name = "roberta-base"

print(f"Loading tokenizer: {model2_name}")
tokenizer2 = AutoTokenizer.from_pretrained(model2_name)

print("Loading model architecture...")
model2 = AutoModelForSequenceClassification.from_pretrained(model2_name, num_labels=2)
model2.to(device)

def tokenize2_function(examples):
    # CRITICAL CHANGE: RoBERTa does not use 'token_type_ids'. 
    # We must explicitly pop or exclude it if your dataset contains it,
    # or ensure our tokenizer doesn't return it.
    return tokenizer2(
        examples['text'], 
        padding="max_length", 
        truncation=True, 
        max_length=256,
        return_token_type_ids=False # <-- ADD THIS LINE
    )

print("Tokenizing training data...")
tokenized2_train = train_dataset.map(tokenize2_function, batched=True)

print("Tokenizing testing data...")
tokenized2_test = test_dataset.map(tokenize2_function, batched=True)

Loading tokenizer: roberta-base
Loading model architecture...


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tokenizing training data...


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing testing data...


Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

## Step 3 Fine-Tuning with Trainer API
Hugging Face provides an incredible class called `Trainer` that handles all the complex PyTorch training loops for us.
We just need to define how we want it to evaluate (e.g., compute accuracy) and define our training hyperparameters.

In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Define training arguments
training_args = TrainingArguments(
    output_dir='../models/bert_sentiment',
    num_train_epochs=3,              # Train for 3 passes over the data
    per_device_train_batch_size=8,   # Small batch size to avoid running out of memory
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    evaluation_strategy="epoch",     # Evaluate at the end of each epoch
    save_strategy="epoch",
    logging_dir='./logs',
    # 2. Learning Rate Optimization
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=0.1,
    weight_decay=0.01
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics
)

print("Starting Fine-Tuning (This will take time depending on your hardware!)...")
trainer.train()

c:\Users\padol\anaconda3\Lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Starting Fine-Tuning (This will take time depending on your hardware!)...


  0%|          | 0/468 [00:00<?, ?it/s]

  0%|          | 0/94 [00:00<?, ?it/s]

{'eval_loss': 0.2586025893688202, 'eval_accuracy': 0.8973333333333333, 'eval_f1': 0.8998699609882965, 'eval_precision': 0.8553770086526576, 'eval_recall': 0.9492455418381345, 'eval_runtime': 15.3059, 'eval_samples_per_second': 98.002, 'eval_steps_per_second': 6.141, 'epoch': 1.0}


  0%|          | 0/94 [00:00<?, ?it/s]

{'eval_loss': 0.2629154324531555, 'eval_accuracy': 0.9106666666666666, 'eval_f1': 0.9106666666666666, 'eval_precision': 0.8858625162127107, 'eval_recall': 0.9368998628257887, 'eval_runtime': 28.7795, 'eval_samples_per_second': 52.12, 'eval_steps_per_second': 3.266, 'epoch': 2.0}


  0%|          | 0/94 [00:00<?, ?it/s]

{'eval_loss': 0.2661421597003937, 'eval_accuracy': 0.9186666666666666, 'eval_f1': 0.9164383561643835, 'eval_precision': 0.9151846785225718, 'eval_recall': 0.9176954732510288, 'eval_runtime': 13.1273, 'eval_samples_per_second': 114.266, 'eval_steps_per_second': 7.161, 'epoch': 3.0}
{'train_runtime': 570.0422, 'train_samples_per_second': 26.314, 'train_steps_per_second': 0.821, 'train_loss': 0.22382113464877137, 'epoch': 3.0}


TrainOutput(global_step=468, training_loss=0.22382113464877137, metrics={'train_runtime': 570.0422, 'train_samples_per_second': 26.314, 'train_steps_per_second': 0.821, 'total_flos': 1970175582535680.0, 'train_loss': 0.22382113464877137, 'epoch': 2.9952})

In [6]:

# Define training arguments
training2_args = TrainingArguments(
    output_dir='../models/roberta_sentiment',
    num_train_epochs=3,              # Train for 3 passes over the data
    per_device_train_batch_size=8,   # Small batch size to avoid running out of memory
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    evaluation_strategy="epoch",     # Evaluate at the end of each epoch
    save_strategy="epoch",
    logging_dir='./logs',
    # 2. Learning Rate Optimization
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=0.1,
    weight_decay=0.01
)

# Initialize the Trainer
trainer2 = Trainer(
    model=model2,
    args=training2_args,
    train_dataset=tokenized2_train,
    eval_dataset=tokenized2_test,
    compute_metrics=compute_metrics
)

print("Starting Fine-Tuning (This will take time depending on your hardware!)...")
trainer2.train()

c:\Users\padol\anaconda3\Lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Starting Fine-Tuning (This will take time depending on your hardware!)...


  0%|          | 0/468 [00:00<?, ?it/s]

  0%|          | 0/94 [00:00<?, ?it/s]

{'eval_loss': 0.2268349975347519, 'eval_accuracy': 0.9173333333333333, 'eval_f1': 0.9111747851002865, 'eval_precision': 0.9535232383808095, 'eval_recall': 0.8724279835390947, 'eval_runtime': 14.776, 'eval_samples_per_second': 101.516, 'eval_steps_per_second': 6.362, 'epoch': 1.0}


  0%|          | 0/94 [00:00<?, ?it/s]

{'eval_loss': 0.23507434129714966, 'eval_accuracy': 0.934, 'eval_f1': 0.93421926910299, 'eval_precision': 0.9059278350515464, 'eval_recall': 0.9643347050754458, 'eval_runtime': 26.3709, 'eval_samples_per_second': 56.881, 'eval_steps_per_second': 3.565, 'epoch': 2.0}


  0%|          | 0/94 [00:00<?, ?it/s]

{'eval_loss': 0.22216840088367462, 'eval_accuracy': 0.9386666666666666, 'eval_f1': 0.9371584699453552, 'eval_precision': 0.9333333333333333, 'eval_recall': 0.9410150891632373, 'eval_runtime': 14.7979, 'eval_samples_per_second': 101.366, 'eval_steps_per_second': 6.352, 'epoch': 3.0}
{'train_runtime': 608.7185, 'train_samples_per_second': 24.642, 'train_steps_per_second': 0.769, 'train_loss': 0.21045357141739282, 'epoch': 3.0}


TrainOutput(global_step=468, training_loss=0.21045357141739282, metrics={'train_runtime': 608.7185, 'train_samples_per_second': 24.642, 'train_steps_per_second': 0.769, 'total_flos': 1970175582535680.0, 'train_loss': 0.21045357141739282, 'epoch': 2.9952})

## Step 4 Evaluate Results
Now we see how well BERT performs on the test set.

In [7]:
print("Evaluating on test set...")
results = trainer.evaluate()
print("\n--- BERT Results ---")
for key, value in results.items():
    print(f"{key}: {value}")

# Save the final model
trainer.save_model('../models/bert_sentiment_final')
tokenizer.save_pretrained('../models/bert_sentiment_final')
print("Model saved successfully!")

Evaluating on test set...


  0%|          | 0/94 [00:00<?, ?it/s]


--- BERT Results ---
eval_loss: 0.2661421597003937
eval_accuracy: 0.9186666666666666
eval_f1: 0.9164383561643835
eval_precision: 0.9151846785225718
eval_recall: 0.9176954732510288
eval_runtime: 15.5682
eval_samples_per_second: 96.35
eval_steps_per_second: 6.038
epoch: 2.9952
Model saved successfully!


In [ ]:
print("Evaluating on test set...")
results2 = trainer2.evaluate()
print("\n--- RoBERTa Results ---")
for key, value in results2.items():
    print(f"{key}: {value}")

# Save the final model
trainer2.save_model('../models/roberta_sentiment_final')
tokenizer2.save_pretrained('../models/roberta_sentiment_final')
print("Model saved successfully!")

Evaluating on test set...


  0%|          | 0/94 [00:00<?, ?it/s]


--- BERT Results ---
eval_loss: 0.22216840088367462
eval_accuracy: 0.9386666666666666
eval_f1: 0.9371584699453552
eval_precision: 0.9333333333333333
eval_recall: 0.9410150891632373
eval_runtime: 16.7632
eval_samples_per_second: 89.482
eval_steps_per_second: 5.608
epoch: 2.9952
Model saved successfully!


## Context Understanding & Sarcasm (Bonus Comparison)
Traditional models struggle with sarcasm because they only look at individual words. BERT looks at the entire sentence context bidirectionally. Let's test it on a tricky sentence!

In [9]:
from transformers import pipeline

# Load our newly trained model into an easy-to-use pipeline
sentiment_pipeline = pipeline("sentiment-analysis", model='../models/bert_sentiment_final', tokenizer='../models/bert_sentiment_final')
sentiment2_pipeline = pipeline("sentiment-analysis", model='../models/roberta_sentiment_final', tokenizer='../models/roberta_sentiment_final')

tricky_sentences = [
    "I absolutely loved wasting two hours of my life on this garbage.", # Sarcasm
    "The movie was not terrible, I actually quite liked it.",          # Negation
    "A masterpiece of terrible writing."                               # Contradictory
]

print("--- BERT Inference on Tricky Sentences ---")
for sentence in tricky_sentences:
    result = sentiment_pipeline(sentence)[0]
    result2 = sentiment2_pipeline(sentence)[0]
    # label_1 usually means positive in our mapping, label_0 means negative.
    # The pipeline outputs LABEL_0 or LABEL_1.
    label_name = "Positive" if result['label'] == 'LABEL_1' else "Negative"
    label_name2 = "Positive" if result2['label'] == 'LABEL_1' else "Negative"
    print(f"Sentence: '{sentence}'")
    print(f"Prediction: {label_name} (Confidence: {result['score']:.4f})\n")
    print(f"Prediction 2: {label_name2} (Confidence: {result2['score']:.4f})\n")


--- BERT Inference on Tricky Sentences ---
Sentence: 'I absolutely loved wasting two hours of my life on this garbage.'
Prediction: Negative (Confidence: 0.8410)

Prediction 2: Negative (Confidence: 0.9910)

Sentence: 'The movie was not terrible, I actually quite liked it.'
Prediction: Positive (Confidence: 0.8491)

Prediction 2: Positive (Confidence: 0.9876)

Sentence: 'A masterpiece of terrible writing.'
Prediction: Negative (Confidence: 0.9819)

Prediction 2: Negative (Confidence: 0.9957)

